In [0]:
import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *
import logging
from datetime import datetime
import time
from requests.auth import HTTPBasicAuth
from delta.tables import DeltaTable

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

spark = SparkSession.builder \
    .appName("API_Posting_Delta_Update") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

df_date = spark.sql("SELECT current_date() as today")
current_date_value = df_date.collect()[0]['today'].strftime('%Y-%m-%d')

CONFIG = {
    "delta_table_path": "jeffdemo.rohit_test.invoice_data_parsed",  
    "api_endpoint": "https://org-pocus-poc-invoiceext-srv.cfapps.us10.hana.ondemand.com/odata/v4/zapi-supplierinvoice-process-srvsample/postInvoice",  
    "auth": {
        "username": "Admin", 
        "password": "Admin"  
    },
    "api_headers": {
        "Content-Type": "application/json",
        "Accept": "application/json",
        "User-Agent": "Spark-API-Client/1.0"
    },
    "field_mapping": {
        ""
        "invoice_number": "SupplierInvoiceIDByInvcgParty",
        "amount": "InvoiceGrossAmount"
    },
    "constant_fields": {
        "CompanyCode": "PHC1",
        "InvoicingParty": "0001000006",
        "DocumentCurrency": "USD",
        "DocumentDate": current_date_value,
        "SupplierInvoiceStatus": "A",
        "PostingDate": current_date_value
    },

    "field_transformations": {
        "DocumentDate": "date_format",
        "PostingDate": "date_format"
    },
    
    "api_date_format": "%Y-%m-%d",

    "batch_size": 100,  
    "max_retries": 3,
    "retry_delay": 5  
}

def transform_field_value(field_name, value, config):
    """
    Transform field values based on configuration
    """
    if value is None:
        return None
    
    transformations = config.get("field_transformations", {})
    transformation_type = transformations.get(field_name)
    
    try:
        if transformation_type == "date_format":
            if isinstance(value, str):
                for fmt in ["%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d %H:%M:%S"]:
                    try:
                        parsed_date = datetime.strptime(value, fmt)
                        return parsed_date.strftime(config["api_date_format"])
                    except ValueError:
                        continue
                return value
            elif hasattr(value, 'strftime'):  
                return value.strftime(config["api_date_format"])
            else:
                return str(value)
                
        elif transformation_type == "decimal_format":
            if isinstance(value, (int, float)):
                return str(value)
            elif isinstance(value, str):
                try:
                    return  str(value)
                except ValueError:
                    return value
            else:
                return  str(value)
                
        else:
            return value
            
    except Exception as e:
        logger.warning(f"Failed to transform field {field_name} with value {value}: {str(e)}")
        return value

def map_delta_record_to_api_payload(record_dict, config):
    """
    Map Delta table record to API payload format
    """
    field_mapping = config.get("field_mapping", {})
    constant_fields = config.get("constant_fields", {})
    api_payload = {}
    
    for constant_field, constant_value in constant_fields.items():
        api_payload[constant_field] = constant_value
        logger.debug(f"Added constant field: {constant_field} = {constant_value}")
        
    for delta_field, api_field in field_mapping.items():
        if delta_field in record_dict:
            original_value = record_dict[delta_field]
            transformed_value = transform_field_value(delta_field, original_value, config)
            api_payload[api_field] = transformed_value
        else:
            logger.warning(f"Field '{delta_field}' not found in record")
            api_payload[api_field] = None
    
    return api_payload

def post_to_api_with_basic_auth(data_payload, config):
    """
    Post data to API endpoint with Basic Authentication and retry logic
    """
    auth = config.get("auth", {})
    username = auth.get("username")
    password = auth.get("password")
    for attempt in range(config["max_retries"]):
        try:
            response = requests.post(
                config["api_endpoint"],
                auth=HTTPBasicAuth(username, password),
                headers=config["api_headers"],
                json=data_payload
            )
            
            if response.status_code == 200:
                return {
                    "status": "SUCCESS",
                    "response_data": response.json() if response.content else {},
                    "error_message": None,
                    "http_status": response.status_code
                }
            elif response.status_code == 201: 
                return {
                    "status": "SUCCESS",
                    "response_data": response.json() if response.content else {},
                    "error_message": None,
                    "http_status": response.status_code
                }
            elif response.status_code == 401:
                return {
                    "status": "AUTH_FAILED",
                    "response_data": None,
                    "error_message": f"Authentication failed: HTTP {response.status_code}",
                    "http_status": response.status_code
                }
            elif response.status_code == 403:
                return {
                    "status": "FORBIDDEN",
                    "response_data": None,
                    "error_message": f"Access forbidden: HTTP {response.status_code}",
                    "http_status": response.status_code
                }
            else:
                return {
                    "status": "FAILED",
                    "response_data": None,
                    "error_message": f"HTTP {response.status_code}: {response.text}",
                    "http_status": response.status_code
                }
                
        except requests.exceptions.Timeout as e:
            error_msg = f"Request timeout (attempt {attempt + 1}): {str(e)}"
            if attempt < config["max_retries"] - 1:
                logger.warning(f"{error_msg}, retrying in {config['retry_delay']} seconds")
                time.sleep(config["retry_delay"])
            else:
                return {
                    "status": "TIMEOUT",
                    "response_data": None,
                    "error_message": error_msg,
                    "http_status": None
                }
                
        except requests.exceptions.ConnectionError as e:
            error_msg = f"Connection error (attempt {attempt + 1}): {str(e)}"
            if attempt < config["max_retries"] - 1:
                logger.warning(f"{error_msg}, retrying in {config['retry_delay']} seconds")
                time.sleep(config["retry_delay"])
            else:
                return {
                    "status": "CONNECTION_ERROR",
                    "response_data": None,
                    "error_message": error_msg,
                    "http_status": None
                }
                
        except requests.exceptions.RequestException as e:
            error_msg = f"Request error (attempt {attempt + 1}): {str(e)}"
            if attempt < config["max_retries"] - 1:
                logger.warning(f"{error_msg}, retrying in {config['retry_delay']} seconds")
                time.sleep(config["retry_delay"])
            else:
                return {
                    "status": "REQUEST_ERROR",
                    "response_data": None,
                    "error_message": error_msg,
                    "http_status": None
                }
    
    return {
        "status": "MAX_RETRIES_EXCEEDED",
        "response_data": None,
        "error_message": "Maximum retry attempts exceeded",
        "http_status": None
    }

def process_record_batch_with_auth(records, config):
    """
    Process a batch of records with Basic Authentication
    """
    results = []
    
    for record in records:
        record_dict = record.asDict()
        record_id = record_dict.get('invoice_number') 
        mapped_payload = map_delta_record_to_api_payload(record_dict, config)
        api_payload =  mapped_payload   
        api_result = post_to_api_with_basic_auth(api_payload, config)
        
        result = {
            "id": record_id,
            "status": api_result["status"],
            "processed_at": datetime.now(),
            "error_message": api_result["error_message"],
            "http_status_code": api_result["http_status"],
            "response_data": json.dumps(api_result["response_data"]) if api_result["response_data"] else None
        }
        
        results.append(result)
        
        logger.info(f"Processed record {record_id}: {api_result['status']} (HTTP: {api_result['http_status']})")
    
    return results

def process_delta_table_with_field_mapping(config):
    """
    Main function to process Delta table records with field mapping, constant fields, and Basic Authentication
    """
    try:
 
        delta_table1 = spark.table(config["delta_table_path"])
        
    
        df = delta_table1.filter(
            (col("post_status").isNull()) | 
            (col("post_status") == "") |
            (col("post_status").isin(["PENDING", "FAILED", "TIMEOUT", "CONNECTION_ERROR", "MAPPING_ERROR"]))
        )
        
        total_records = df.count()
        logger.info(f"Found {total_records} records to process")
        
        if total_records == 0:
            logger.info("No records to process")
            return {
                "total_processed": 0,
                "successful": 0,
                "errors": 0,
                "auth_failures": 0,
                "mapping_errors": 0
            }
        
    
        processed_count = 0
        success_count = 0
        error_count = 0
        auth_failure_count = 0
        mapping_error_count = 0
        
        records = df.collect()
        
        for i in range(0, len(records), config["batch_size"]):
            batch = records[i:i + config["batch_size"]]
            batch_num = i//config["batch_size"] + 1
            logger.info(f"Processing batch {batch_num}, records {i+1} to {__builtins__.min(i+config['batch_size'], len(records))}")
            
            batch_results = process_record_batch_with_auth(batch, config)
            
            for result in batch_results:
                try:
                    table_name = config["delta_table_path"]
                    deltaTable1 = DeltaTable.forName(spark, table_name)
                    deltaTable1.update(
                        condition = col("invoice_number") == result["id"],
                        set = {
                            "post_status": lit(result["status"]),
                            "post_error": lit(result["error_message"])
                        }
                    )
                    
                    processed_count += 1
                    if result["status"] == "SUCCESS":
                        success_count += 1
                    elif result["status"] in ["AUTH_FAILED", "FORBIDDEN"]:
                        auth_failure_count += 1
                    elif result["status"] == "MAPPING_ERROR":
                        mapping_error_count += 1
                    else:
                        error_count += 1
                        
                except Exception as e:
                    logger.error(f"Failed to update record {result['id']}: {str(e)}")
            
            time.sleep(1)
        
        logger.info(f"Processing completed:")
        logger.info(f"  Total processed: {processed_count}")
        logger.info(f"  Successful: {success_count}")
        logger.info(f"  Authentication failures: {auth_failure_count}")
        logger.info(f"  Mapping errors: {mapping_error_count}")
        logger.info(f"  Other errors: {error_count}")
        
        return {
            "total_processed": processed_count,
            "successful": success_count,
            "errors": error_count,
            "auth_failures": auth_failure_count,
            "mapping_errors": mapping_error_count
        }
        
    except Exception as e:
        logger.error(f"Error in main processing function: {str(e)}")
        raise
    
process_delta_table_with_field_mapping(CONFIG)